# Value dependence of the reference-attention oracle (twelve carriers), Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16 weights. Teacher-forced on the frozen twelve-line plan prompt plus the reference's saved plan, exactly as in the replay run. Same nine cases. Oracle diagnostic, not a method. No adapter.

Branches:
- **tk_tv**: transferred cache, ordinary attention
- **fk_fv**: reference, ordinary attention (never edited)
- **tk_tv_refattn**: normal oracle; should reproduce the replay run's decisions
- **tk_tv_refattn_vzero**: same oracle; target result-region values set to zero
- **tk_tv_refattn_vperm**: same oracle; target result-region value rows permuted with one fixed derangement (same for every layer and head)

Keys, support, positions, masks, token IDs and cache length unchanged. Under replacement the target row's own queries and keys are unused, so the transferred cache acts on it only through values.

Per-case checks: edits touch only result values; reference row unchanged within tolerance in every pair; replacement hits every layer; input caches unchanged; normal oracle reproduces the replay run. Stops at the first failing case.

**Cell 4** runs the first case; **Cell 5** the remaining eight (resumes).

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, and the folder `results_stage_plan4_refattn12/` (needs `a_artifacts_12.jsonl`, `recon12_check_summary.json`, `kvmix12_evaluations.jsonl`, `evaluations.jsonl`).

Download `/kaggle/working/results_stage_plan4_refval12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_refval12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_refval12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_refval12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_refval12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for name in ("a_artifacts_12.jsonl", "recon12_check_summary.json", "kvmix12_evaluations.jsonl", "evaluations.jsonl"):
    print("results_stage_plan4_refattn12", name, (REPO / "results_stage_plan4_refattn12" / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_refval12. First case, then the rest.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refval12", "--split", "plan4_timing12", "--limit", "1"])
print("refval12 first case exit", rc)
path = WORK / "results_stage_plan4_refval12" / "case_checks.jsonl"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_refval12" / "run" / "case_checks.jsonl"
rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
for row in rows:
    print(row["example_id"], "ok", row["ok"], "n_result", row["n_result"])
    print(json.dumps(row["checks"], indent=1)[:4000])
if not all(r["ok"] for r in rows):
    raise RuntimeError("Check failed. Do not run the remaining cases.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refval12", "--split", "plan4_timing12"])
print("refval12 exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_refval12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_refval12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
print("cases", data["n_cases"], "of", data["n_expected_cases"], "| instrumentation_ok", data["instrumentation_ok"])
print("oracle reproduces replay run", data["oracle_reproduces_refattn12"])
keys = ("labels_correct", "n_labels", "labels_agree_ref", "ineligible_marked_eligible", "eligible_marked_ineligible",
        "cases_all_labels_correct", "mean_kl_vs_reference", "mean_label_kl_vs_reference", "mean_argmax_agree_reference")
for b, block in data["branches"].items():
    print(b, {k: block.get(k) for k in keys})
print("\nrecovery retained", data["recovery_retained"])
print("decision change vs oracle", data["decision_change_vs_oracle"])
for name, block in data["paired_by_case"].items():
    c = block["labels_correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")})